In [2]:
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# Set up paths based on your file architecture
data_root = os.path.join("..", "data", "splits")
rotations = ["rotation_0", "rotation_1", "rotation_2", "rotation_3"]

# Setup device for PyTorch (uses GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load frozen DeBERTa model and tokenizer safely
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Force safetensors loading to completely bypass the torch.load security restriction
model = AutoModel.from_pretrained(model_name, use_safetensors=True)

model.to(device)
model.eval() # Set to evaluation mode to freeze dropout/batchnorm


# Helper function to extract embeddings in batches
def get_deberta_embeddings(texts, batch_size=16, max_length=384):
    all_embeddings = []
    
    # Process texts in batches to prevent Out-Of-Memory (OOM) errors
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Features"):
        batch_texts = texts[i : i + batch_size].tolist()
        
        # Tokenize batch
        encoded = tokenizer(
            batch_texts, 
            padding=True, 
            truncation=True, 
            max_length=max_length, 
            return_tensors="pt"
        ).to(device)
        
        # Disable gradient calculation for purely extracting features
        with torch.no_grad():
            outputs = model(**encoded)
            
            # Extract the [CLS] token representation (the 0th token of the last hidden state)
            # This serves as our pooled review vector 'h'
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_embeddings.cpu().numpy())
            
    return np.vstack(all_embeddings)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# List to hold the final results
all_results = []

for rot in rotations:
    print(f"\n--- Processing {rot} ---")
    path = os.path.join(data_root, rot)
    
    # Load datasets
    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_in_df = pd.read_csv(os.path.join(path, "test_indist.csv"))
    test_cross_df = pd.read_csv(os.path.join(path, "test_crossgen.csv"))
    
    # Extract text and handle NaNs
    train_text = train_df['text'].astype(str).fillna("")
    test_in_text = test_in_df['text'].astype(str).fillna("")
    test_cross_text = test_cross_df['text'].astype(str).fillna("")
    
    y_train = train_df['label'].values
    y_test_in = test_in_df['label'].values
    y_test_cross = test_cross_df['label'].values

    # Extract DeBERTa embeddings
    print("Train dataset:")
    x_train = get_deberta_embeddings(train_text)
    print("In-Distribution Test dataset:")
    x_test_in = get_deberta_embeddings(test_in_text)
    print("Cross-Generator Test dataset:")
    x_test_cross = get_deberta_embeddings(test_cross_text)

    # Train Logistic Regression on the frozen embeddings
    print("Fitting Logistic Regression...")
    clf = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
    clf.fit(x_train, y_train)

    # Evaluate In-Distribution
    prob_in = clf.predict_proba(x_test_in)[:, 1]
    pred_in = (prob_in >= 0.5).astype(int)
    f1_in = f1_score(y_test_in, pred_in)
    auc_in = roc_auc_score(y_test_in, prob_in)

    # Evaluate Cross-Generator
    prob_cross = clf.predict_proba(x_test_cross)[:, 1]
    pred_cross = (prob_cross >= 0.5).astype(int)
    f1_cross = f1_score(y_test_cross, pred_cross)
    auc_cross = roc_auc_score(y_test_cross, prob_cross)

    # Calculate per-generator scores using the fixed logic
    test_cross_df['preds'] = pred_cross
    gen_scores = {}
    human_subset = test_cross_df[test_cross_df['label'] == 0]
    
    for gen in test_cross_df['generator'].unique():
        if pd.isna(gen) or gen.lower() == 'human':
            continue
            
        ai_subset = test_cross_df[test_cross_df['generator'] == gen]
        if not ai_subset.empty and ai_subset['label'].iloc[0] == 1:
            eval_subset = pd.concat([ai_subset, human_subset])
            gen_f1 = f1_score(eval_subset['label'], eval_subset['preds'])
            gen_scores[f"{gen}_F1"] = gen_f1

    # Store results
    res = {
        "Rotation": rot,
        "In-Dist F1": f1_in,
        "In-Dist AUC": auc_in,
        "Cross-Gen F1": f1_cross,
        "Cross-Gen AUC": auc_cross,
        "Gap (F1)": f1_in - f1_cross
    }
    res.update(gen_scores)
    all_results.append(res)
    
    print(f"Completed {rot} | F1 Gap: {(f1_in - f1_cross):.4f}")


--- Processing rotation_0 ---
Train dataset:


Extracting Features: 100%|██████████| 750/750 [01:07<00:00, 11.19it/s]


In-Distribution Test dataset:


Extracting Features: 100%|██████████| 94/94 [00:08<00:00, 10.86it/s]


Cross-Generator Test dataset:


...

In-Distribution Test dataset:


Extracting Features: 100%|██████████| 94/94 [00:08<00:00, 11.05it/s]


Cross-Generator Test dataset:


Extracting Features: 100%|██████████| 312/312 [00:30<00:00, 10.23it/s]


Fitting Logistic Regression...
Completed rotation_3 | F1 Gap: 0.0022


In [5]:
# Convert list of results into a pandas DataFrame
results_df = pd.DataFrame(all_results)

# Calculate averages
averages = results_df.mean(numeric_only=True).to_dict()
averages["Rotation"] = "AVERAGE"
summary_table = pd.concat([results_df, pd.DataFrame([averages])], ignore_index=True)

# Save results
summary_table.to_csv("deberta_baseline_results.csv", index=False)

print("\n--- DeBERTa (FROZEN) + LOGREG BASELINE SUMMARY ---")
display(summary_table.round(4))


--- DeBERTa (FROZEN) + LOGREG BASELINE SUMMARY ---


,Rotation,In-Dist F1,In-Dist AUC,Cross-Gen F1,Cross-Gen AUC,Gap (F1),gpt5mini_F1,deepseek_F1,gemma_F1,qwen_F1
0,rotation_0,0.9601,0.9927,0.9678,0.9964,-0.0078,0.9678,NaN,NaN,NaN
1,rotation_1,0.9735,0.9975,0.9146,0.9762,0.0589,NaN,0.9146,NaN,NaN
2,rotation_2,0.9634,0.9946,0.9287,0.9856,0.0347,NaN,NaN,0.9287,NaN
3,rotation_3,0.9657,0.9953,0.9635,0.9945,0.0022,NaN,NaN,NaN,0.9635
4,AVERAGE,0.9657,0.9950,0.9437,0.9882,0.0220,0.9678,0.9146,0.9287,0.9635
